# Day 2 — QLoRA Training
**Goal:** Fine-tune `mistral-7b-instruct-v0.3` using QLoRA (r=16, α=32) and log everything to MLflow.

**Expected:** ~2–4 hours on RTX 5090 | train_loss ≈ 1.0–1.3 | eval_loss ≈ 0.9–1.1

**Outputs:**
- `adapters/` — LoRA adapter weights
- `mlruns/` — MLflow experiment logs

In [1]:
import sys, os
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
os.chdir(PROJECT_ROOT)

print(f'Project root: {PROJECT_ROOT}')

Project root: /home/ubuntu/Fine-tuning/NyayaGPT


In [2]:
# ── GPU check ──────────────────────────────────────────────────────────────────
import torch

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    vram_gb = props.total_memory / 1e9
    print(f'✓ GPU: {props.name}')
    print(f'  VRAM: {vram_gb:.1f} GB')
    print(f'  SM:   {props.major}.{props.minor}')
    print(f'  BF16: {torch.cuda.is_bf16_supported()}')
    
    if vram_gb < 10:
        print('⚠  Less than 10 GB VRAM — reduce batch size or gradient accumulation')
    else:
        print(f'✓  {vram_gb:.0f} GB VRAM is sufficient for Mistral 7B QLoRA')
else:
    print('✗ No GPU detected — training will fail')

✓ GPU: NVIDIA GeForce RTX 5090
  VRAM: 34.2 GB
  SM:   12.0
  BF16: True
✓  34 GB VRAM is sufficient for Mistral 7B QLoRA


In [3]:
# ── Verify datasets exist ─────────────────────────────────────────────────────
from nyaya_pipeline import config

for path, label in [(config.TRAIN_JSONL, 'train'), (config.EVAL_JSONL, 'eval')]:
    if path.exists():
        n = sum(1 for _ in open(path, encoding='utf-8'))
        print(f'✓ {label}: {n} samples → {path}')
    else:
        print(f'✗ {label} not found: {path}')
        print('  Run notebook 01 first!')

✓ train: 1521 samples → /home/ubuntu/Fine-tuning/NyayaGPT/output/train.jsonl
✓ eval: 169 samples → /home/ubuntu/Fine-tuning/NyayaGPT/output/eval.jsonl


## Training Configuration

| Parameter | Value | Notes |
|-----------|-------|-------|
| Base model | mistral-7b-instruct-v0.3-bnb-4bit | Pre-quantized Unsloth variant |
| LoRA r | 16 | Rank — higher = more capacity |
| LoRA α | 32 | = 2×r (standard scaling) |
| Epochs | 3 | Sufficient for ~1,350 samples |
| LR | 2e-4 | QLoRA canonical value |
| Batch size | 2 | Per-device (× 4 grad accum = eff. 8) |
| Precision | BF16 | RTX 5090 supports BF16 natively |

In [4]:
# ── Optional: override config for quick test ──────────────────────────────────
# For a 5-minute smoke test, uncomment these:
#
# os.environ['NY_EPOCHS'] = '1'
# os.environ['NY_TRAIN_BATCH'] = '1'
# os.environ['NY_GRAD_ACCUM'] = '2'
# 
# Reload config after env-var changes
# import importlib
# from nyaya_pipeline import config as cfg
# importlib.reload(cfg)

print('Config:')
print(f'  model        : {config.STUDENT_MODEL_NAME}')
print(f'  epochs       : {config.NUM_TRAIN_EPOCHS}')
print(f'  lr           : {config.LEARNING_RATE}')
print(f'  batch_size   : {config.TRAIN_BATCH_SIZE}')
print(f'  grad_accum   : {config.GRAD_ACCUM_STEPS}')
print(f'  lora_r       : {config.LORA_R}')
print(f'  lora_alpha   : {config.LORA_ALPHA}')
print(f'  max_seq_len  : {config.STUDENT_MAX_SEQ_LEN}')
print(f'  adapter_dir  : {config.ADAPTER_DIR}')
print(f'  mlflow_uri   : {config.MLFLOW_URI}')

Config:
  model        : unsloth/mistral-7b-instruct-v0.3-bnb-4bit
  epochs       : 2
  lr           : 0.0002
  batch_size   : 2
  grad_accum   : 4
  lora_r       : 16
  lora_alpha   : 32
  max_seq_len  : 2048
  adapter_dir  : /home/ubuntu/Fine-tuning/NyayaGPT/adapters
  mlflow_uri   : /home/ubuntu/Fine-tuning/NyayaGPT/mlruns


In [5]:
# ── Run QLoRA training ────────────────────────────────────────────────────────
from src.nyaya_pipeline.trainer import train

adapter_path = train(
    mlflow_run_name='mistral-7b-qlora-v1',
)

print(f'\n✓ Adapter saved to: {adapter_path}')

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
2026-04-19 14:46:07,258 [INFO] src.nyaya_pipeline.trainer: Loading datasets …


[src.nyaya_pipeline.trainer|INFO]Loading datasets …


2026-04-19 14:46:07,276 [INFO] src.nyaya_pipeline.trainer:   train: 1521 samples | eval: 169 samples


[src.nyaya_pipeline.trainer|INFO]  train: 1521 samples | eval: 169 samples


2026-04-19 14:46:07,277 [INFO] src.nyaya_pipeline.trainer: Loading base model: unsloth/mistral-7b-instruct-v0.3-bnb-4bit


...

Epoch,Training Loss,Validation Loss
1,0.686000,0.704199
2,0.336500,0.486596


Unsloth: Not an error, but MistralForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


2026-04-19 14:55:43,566 [INFO] src.nyaya_pipeline.trainer: Training complete in 539.5 s | train_loss=0.7504 | eval_loss=0.4866


[src.nyaya_pipeline.trainer|INFO]Training complete in 539.5 s | train_loss=0.7504 | eval_loss=0.4866



✓ Adapter saved to: /home/ubuntu/Fine-tuning/NyayaGPT/adapters


In [6]:
# ── Verify adapter files ──────────────────────────────────────────────────────
import os

adapter_dir = config.ADAPTER_DIR
print(f'Contents of {adapter_dir}:')
for f in sorted(adapter_dir.rglob('*')):
    if f.is_file():
        size_mb = f.stat().st_size / 1e6
        print(f'  {f.relative_to(adapter_dir)}  ({size_mb:.1f} MB)')

Contents of /home/ubuntu/Fine-tuning/NyayaGPT/adapters:
  README.md  (0.0 MB)
  adapter_config.json  (0.0 MB)
  adapter_model.safetensors  (83.9 MB)
  chat_template.jinja  (0.0 MB)
  checkpoints/README.md  (0.0 MB)
  checkpoints/checkpoint-191/README.md  (0.0 MB)
  checkpoints/checkpoint-191/adapter_config.json  (0.0 MB)
  checkpoints/checkpoint-191/adapter_model.safetensors  (167.8 MB)
  checkpoints/checkpoint-191/chat_template.jinja  (0.0 MB)
  checkpoints/checkpoint-191/optimizer.pt  (85.7 MB)
  checkpoints/checkpoint-191/rng_state.pth  (0.0 MB)
  checkpoints/checkpoint-191/scheduler.pt  (0.0 MB)
  checkpoints/checkpoint-191/special_tokens_map.json  (0.0 MB)
  checkpoints/checkpoint-191/tokenizer.json  (3.7 MB)
  checkpoints/checkpoint-191/tokenizer.model  (0.6 MB)
  checkpoints/checkpoint-191/tokenizer_config.json  (0.1 MB)
  checkpoints/checkpoint-191/trainer_state.json  (0.0 MB)
  checkpoints/checkpoint-191/training_args.bin  (0.0 MB)
  checkpoints/checkpoint-382/README.md  (0.0 

In [7]:
# ── View MLflow experiments ───────────────────────────────────────────────────
import mlflow

mlflow.set_tracking_uri(config.MLFLOW_URI)
runs = mlflow.search_runs(experiment_names=[config.MLFLOW_EXPERIMENT_TRAIN])

if not runs.empty:
    cols = ['run_id', 'params.base_model', 'params.num_epochs',
            'metrics.train_loss', 'metrics.eval_loss', 'metrics.training_time_secs']
    existing = [c for c in cols if c in runs.columns]
    print(runs[existing].to_string(index=False))
else:
    print('No MLflow runs found yet')

                          run_id                         params.base_model params.num_epochs  metrics.train_loss  metrics.eval_loss  metrics.training_time_secs
3a6691ed0f1943378424211d0b37d535 unsloth/mistral-7b-instruct-v0.3-bnb-4bit                 2            0.750388           0.486596                  539.489019
c5746126d74b44f4b1ffd229857f8aa7 unsloth/mistral-7b-instruct-v0.3-bnb-4bit                 3            0.589785           0.486697                  813.610874


In [8]:
# ── Quick inference test ──────────────────────────────────────────────────────
from nyaya_pipeline.infer import generate

test_questions = [
    'What is Section 302 of the Indian Penal Code?',
    'Explain the right to life under Article 21 of the Indian Constitution.',
]

for q in test_questions:
    print(f'Q: {q}')
    resp = generate(q, max_new_tokens=150)
    print(f'A: {resp}')
    print()

Q: What is Section 302 of the Indian Penal Code?
2026-04-19 14:57:31,250 [INFO] nyaya_pipeline.infer: Loading model: unsloth/mistral-7b-instruct-v0.3-bnb-4bit (adapter=/home/ubuntu/Fine-tuning/NyayaGPT/adapters)


[nyaya_pipeline.infer|INFO]Loading model: unsloth/mistral-7b-instruct-v0.3-bnb-4bit (adapter=/home/ubuntu/Fine-tuning/NyayaGPT/adapters)


==((====))==  Unsloth 2026.3.8: Fast Mistral patching. Transformers: 4.56.2.
   \\   /|    NVIDIA GeForce RTX 5090. Num GPUs = 1. Max memory: 31.842 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
A: **Answer**

Section 302 of the Indian Penal Code (IPC) deals with murder. It states that whoever commits the offence of murder—that is, causes the death of a person with intent to kill—is liable for punishment under this section. The punishment for murder under Section 302 is death or imprisonment for life, as prescribed by law.

Q: Explain the right to life under Article 21 of the Indian Constitution.
A: **Answer**

The right to life is a fundamental right protected by Article 21 of the Indian Constitution. It guarantees t

## ✅ Day 2 Complete

**Check MLflow UI:**
```bash
mlflow ui --backend-store-uri mlruns/ --port 5000
# Open: http://localhost:5000
```

**Next:** Run `03_evaluation.ipynb`